# MLP em PyTorch para Churn (classificação binária)

## Objetivo
Montar, do zero, uma **rede neural feedforward (MLP)** com PyTorch para prever churn usando o mesmo dataset tratado do EDA (`Telco_customer_churn_ready.csv`).

Neste notebook você verá de forma direta:
- **Arquitetura:** camadas `Linear` + ativação na camada oculta.
- **Função de ativação:** **ReLU** (comentada no código como escolha didática).
- **Função de perda (loss):** **BCEWithLogitsLoss** (adequada para classificação binária com um único neurônio de saída).

O foco é **aprendizado**, não um modelo de produção: poucas épocas, arquitetura pequena e métricas simples no final.

## O que será feito (passo a passo) e por quê
1. Carregar `Telco_customer_churn_ready.csv` (mesmo padrão do notebook de baselines).
   - Por quê: dados já limpos e codificados, comparáveis ao restante do projeto.
2. Separar `X` e `y`, dividir treino/teste com estratificação.
   - Por quê: avaliar generalização e manter proporção de churn.
3. Padronizar features com `StandardScaler` **apenas no treino** (e aplicar o mesmo no teste).
   - Por quê: redes neurais costumam treinar melhor com entradas na mesma escala.
4. Definir a classe `MLPChurn` (`torch.nn.Module`).
   - Por quê: encapsular pesos, bias e o fluxo `forward`.
5. Usar `BCEWithLogitsLoss` e otimizador **Adam**.
   - Por quê: loss estável para probabilidades e otimizador simples de usar em sala.
6. Loop de treino curto e avaliação com acurácia no conjunto de teste.
   - Por quê: fechar o ciclo básico *dados → modelo → loss → treino → métrica*.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

# Semente para reprodutibilidade (numpy e PyTorch).
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Mesma convenção do notebook 02: procurar o CSV tratado na raiz ou um nível acima.
READY_PATH_CANDIDATES = [
    Path("../data/Telco_customer_churn_ready.csv"),
    Path("data/Telco_customer_churn_ready.csv"),
]
DATA_PATH = next((p for p in READY_PATH_CANDIDATES if p.exists()), None)

assert DATA_PATH is not None, (
    "Arquivo tratado do EDA não encontrado. Gere Telco_customer_churn_ready.csv no notebook de EDA."
)
print(f"Arquivo de entrada encontrado: {DATA_PATH.resolve()}")

Arquivo de entrada encontrado: C:\Users\azvef\Projeto FIAP 1\9mlet-tech-challenge-1-churn-prevision\data\Telco_customer_churn_ready.csv


In [2]:
# Carrega o DataFrame e monta X (features) e y (alvo binário).
TARGET = "Churn Value"
df = pd.read_csv(DATA_PATH)

assert TARGET in df.columns, f"Coluna alvo não encontrada: {TARGET}"
X = df.drop(columns=[TARGET])
y = df[TARGET].astype(int)

# Converte para float32: facilita compatibilidade com tensores e StandardScaler.
X_np = X.astype(np.float32).to_numpy()
y_np = y.to_numpy(dtype=np.float32)

# Split estratificado: mesma ideia do baseline sklearn.
X_train, X_test, y_train, y_test = train_test_split(
    X_np,
    y_np,
    test_size=0.2,
    random_state=SEED,
    stratify=y_np,
)

# Ajusta o scaler só no treino; aplica a mesma transformação no teste (evita vazamento de informação).
scaler = StandardScaler()
# StandardScaler devolve float64; convertemos para float32 para alinhar com os tensores PyTorch.
X_train_s = scaler.fit_transform(X_train).astype(np.float32)
X_test_s = scaler.transform(X_test).astype(np.float32)

n_features = X_train_s.shape[1]
print(f"Amostras treino: {len(X_train_s)} | teste: {len(X_test_s)} | features: {n_features}")

Amostras treino: 5634 | teste: 1409 | features: 1162


In [3]:
# TensorDataset + DataLoader: PyTorch consome dados em mini-batches.
train_ds = TensorDataset(
    torch.from_numpy(X_train_s),
    torch.from_numpy(y_train).unsqueeze(1),
)
test_ds = TensorDataset(
    torch.from_numpy(X_test_s),
    torch.from_numpy(y_test).unsqueeze(1),
)

BATCH_SIZE = 128
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

In [4]:
class MLPChurn(nn.Module):
    """Perceptron multicamadas simples: entrada → oculta (ReLU) → uma saída (logit)."""

    def __init__(self, input_dim: int, hidden_dim: int = 64):
        super().__init__()
        # Camada 1: combinação linear de todas as features.
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        # ReLU introduz não-linearidade; sem ela, duas camadas lineares colapsariam em uma só.
        self.relu = nn.ReLU()
        # Camada de saída: um neurônio = um logit por amostra (antes do sigmóide).
        self.fc2 = nn.Linear(hidden_dim, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x


# Instancia o modelo com o número de colunas do dataset.
model = MLPChurn(input_dim=n_features, hidden_dim=64)
print(model)

MLPChurn(
  (fc1): Linear(in_features=1162, out_features=64, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=64, out_features=1, bias=True)
)


In [5]:
# BCEWithLogitsLoss = sigmóide + binary cross-entropy numérica estável.
# A saída da rede são logits; a loss aplica sigmoid internamente — não usar sigmoid manual antes.
criterion = nn.BCEWithLogitsLoss()

# Adam: taxa de aprendizado fixa pequena, típica para começar.
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

N_EPOCHS = 25
model.train()
for epoch in range(1, N_EPOCHS + 1):
    running_loss = 0.0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * xb.size(0)
    epoch_loss = running_loss / len(train_loader.dataset)
    if epoch == 1 or epoch % 5 == 0 or epoch == N_EPOCHS:
        print(f"Época {epoch:02d}/{N_EPOCHS} — loss média (treino): {epoch_loss:.4f}")

Época 01/25 — loss média (treino): 0.5655
Época 05/25 — loss média (treino): 0.3073
Época 10/25 — loss média (treino): 0.2641
Época 15/25 — loss média (treino): 0.2300
Época 20/25 — loss média (treino): 0.1930
Época 25/25 — loss média (treino): 0.1591


In [6]:
# Avaliação: sem gradientes; logits → probabilidade com sigmóide → limiar 0.5.
model.eval()
all_probs = []
all_true = []
with torch.no_grad():
    for xb, yb in test_loader:
        logits = model(xb)
        probs = torch.sigmoid(logits)
        all_probs.append(probs.numpy())
        all_true.append(yb.numpy())

y_prob = np.vstack(all_probs).ravel()
y_true = np.vstack(all_true).ravel().astype(int)
y_pred = (y_prob >= 0.5).astype(int)

acc = accuracy_score(y_true, y_pred)
print(f"Acurácia no teste (limiar 0.5): {acc:.4f}")
print("Dica: compare com DummyClassifier / Regressão Logística no notebook 02 para contextualizar o resultado.")

Acurácia no teste (limiar 0.5): 0.7417
Dica: compare com DummyClassifier / Regressão Logística no notebook 02 para contextualizar o resultado.


## Integração com `src/churn` (refatoração da Etapa 3)

A célula abaixo mostra como reutilizar o novo módulo de engenharia sem remover o fluxo didático deste notebook. Assim, o notebook vira consumidor do código em `src/`.

In [ ]:
from pathlib import Path
import sys

# Adiciona src ao path somente para execução local no notebook.
project_root = Path.cwd().resolve().parent
sys.path.insert(0, str(project_root / "src"))

from churn.data.io import load_features_target
from churn.pipelines.churn_pipeline import build_churn_pipeline

X_df, y_series = load_features_target(path=DATA_PATH)
sk_pipeline = build_churn_pipeline(X_df, estimator_name="logistic_regression")

print("Pipeline sklearn montado com sucesso:")
print(sk_pipeline)